## Step 8: Data Persistence (File Formats)

In this notebook, we focus on writing data to storage using different file formats.

We begin by reloading the dataset to ensure the notebook is self-contained and reproducible. This is a best practice in data engineering, as each step in the pipeline should be independently executable.

In [0]:
df = spark.read.csv(
    "/Volumes/week4_catalog/default/week4_volume/sales_data.csv",
    header=True,
    inferSchema=True
)

from pyspark.sql.functions import col
df_with_tip_pct = df.withColumn(
    "tip_pct",
    col("tip") / col("total_bill")
)

### Writing Data in Parquet Format

In this step, we write the transformed dataset to storage using the Parquet format.

Parquet is a columnar storage format optimized for analytical workloads. It provides efficient compression and faster query performance compared to row-based formats like CSV.

This serves as a baseline before introducing Delta Lake, which builds on top of Parquet.

In [0]:
df_with_tip_pct.write.mode("overwrite").parquet(
  "/Volumes/week4_catalog/default/week4_volume/parquet_output"
)

### Reading Parquet Data for Validation

After writing the dataset in Parquet format, we read it back to validate that the data was stored correctly.

This step ensures:
- the write operation was successful
- schema and data integrity are preserved
- the output path is correct

Validating written data is an important best practice in data engineering pipelines.

In [0]:
parquet_df = spark.read.parquet(
    "/Volumes/week4_catalog/default/week4_volume/parquet_output"
)

parquet_df.show()
parquet_df.printSchema()

+----------+----+------+------+---+------+----+-------------------+
|total_bill| tip|   sex|smoker|day|  time|size|            tip_pct|
+----------+----+------+------+---+------+----+-------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|0.05944673337257211|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|0.16054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|0.16658733936220846|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2| 0.1397804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|0.14680764538430255|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|0.18623962040332148|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|0.22805017103762829|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|0.11607142857142858|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|0.13031914893617022|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2| 0.2185385656292287|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2| 0.1665043816942551|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|0

### Parquet Validation Results

The Parquet dataset was successfully written and read back.

Observations:
- The schema is preserved correctly, including data types for all columns
- The derived column `tip_pct` is present and correctly computed
- The data matches the original dataset

This confirms that Parquet provides reliable storage for structured data and maintains both schema and data integrity.

### Writing Data in Delta Format

In this step, we write the dataset as a Delta table.

Delta Lake extends Parquet by adding a transaction log that enables:
- ACID transactions
- schema enforcement
- versioning (time travel)

Unlike Parquet, which stores only files, Delta stores both data and metadata, allowing reliable and consistent data operations.

In [0]:
df_with_tip_pct.write.mode("overwrite").format("delta").saveAsTable(
    "week4_catalog.default.tips_delta"
)

### Querying the Delta Table

After writing the dataset as a Delta table, we query it using Spark SQL.

This confirms that:
- the table is registered in Unity Catalog
- the data is accessible via SQL
- the dataset can be used in downstream analytics

This demonstrates the advantage of Delta tables over raw file storage formats.

In [0]:
spark.sql("""
          SELECT *
          FROM week4_catalog.default.tips_delta
          LIMIT 10
          """).show()

+----------+----+------+------+---+------+----+-------------------+
|total_bill| tip|   sex|smoker|day|  time|size|            tip_pct|
+----------+----+------+------+---+------+----+-------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|0.05944673337257211|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|0.16054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|0.16658733936220846|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2| 0.1397804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|0.14680764538430255|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|0.18623962040332148|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|0.22805017103762829|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|0.11607142857142858|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|0.13031914893617022|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2| 0.2185385656292287|
+----------+----+------+------+---+------+----+-------------------+



### Inspecting Delta Table Metadata

Delta tables maintain a transaction log that records all operations performed on the table.

We use the DESCRIBE DETAIL command to inspect metadata such as:
- table location
- file format
- number of files
- table size

This demonstrates that Delta tables include structured metadata in addition to raw data files.

In [0]:
spark.sql(
    """
    DESCRIBE DETAIL week4_catalog.default.tips_delta
    """
).show()

+------+--------------------+--------------------+-----------+--------+--------------------+-------------------+----------------+-----------------+--------+-----------+--------------------+----------------+----------------+--------------------+--------------------+-------------+
|format|                  id|                name|description|location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|          properties|minReaderVersion|minWriterVersion|       tableFeatures|          statistics|clusterByAuto|
+------+--------------------+--------------------+-----------+--------+--------------------+-------------------+----------------+-----------------+--------+-----------+--------------------+----------------+----------------+--------------------+--------------------+-------------+
| delta|ee7432c3-e788-406...|week4_catalog.def...|       NULL|        |2026-04-27 09:18:...|2026-04-27 09:18:40|              []|               []|       1|    

### Delta Table Metadata Analysis

The output of DESCRIBE DETAIL confirms that the dataset is stored as a Delta table.

Key observations:
- The format is "delta", confirming that the table includes a transaction log
- The dataset is stored in a single file, as expected for a small dataset
- The table is managed by Unity Catalog, so the storage location is abstracted
- Delta-specific features such as append-only behavior are enabled

This demonstrates that Delta tables provide more than just storage—they include metadata and transaction capabilities that enable reliable data management.

In [0]:
spark.sql(
    """
    DESCRIBE HISTORY week4_catalog.default.tips_delta
    """
).show(truncate=False)

+-------+-------------------+--------------+-----------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName               |operation                        |operationParameters                                                                                                                                                                             |job |notebook          |queryHistoryStatementId             |clusterId       

### Delta Table History

The DESCRIBE HISTORY command shows the transaction log of the Delta table.

Observations:
- The table currently has version 0, representing the initial write operation
- The operation type is CREATE OR REPLACE TABLE AS SELECT, indicating the table was created from a DataFrame
- The table contains 244 rows
- Each operation is timestamped, enabling version tracking

This demonstrates that Delta Lake maintains a full history of changes, enabling data versioning and auditability.

### Time Travel Validation

Querying the Delta table using VERSION AS OF 0 returns the same data as the current table.

This is expected because the table currently has only one version (version 0), representing the initial write operation.

Time travel becomes more useful when multiple versions exist, allowing comparison between different states of the dataset.

In [0]:
spark.sql(
    """
    SELECT * 
    FROM week4_catalog.default.tips_delta
    VERSION AS OF 0
    LIMIT 10
    """
).show()

+----------+----+------+------+---+------+----+-------------------+
|total_bill| tip|   sex|smoker|day|  time|size|            tip_pct|
+----------+----+------+------+---+------+----+-------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|0.05944673337257211|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|0.16054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|0.16658733936220846|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2| 0.1397804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|0.14680764538430255|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|0.18623962040332148|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|0.22805017103762829|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|0.11607142857142858|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|0.13031914893617022|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2| 0.2185385656292287|
+----------+----+------+------+---+------+----+-------------------+



### Creating a New Version of the Delta Table

In this step, we perform an update operation on the Delta table to create a new version.

We modify the `tip` column by increasing its value by 10%. This simulates a data update scenario.

Delta Lake tracks all changes as new versions in the transaction log, allowing us to:
- maintain a full history of the dataset
- compare different versions of the data
- ensure ACID-compliant updates

This operation will create a new version of the table (version 1).

In [0]:
spark.sql(
    """
    UPDATE week4_catalog.default.tips_delta
    SET tip = tip*1.1
    """
)

DataFrame[num_affected_rows: bigint]

### Update Result

The update operation was successfully executed, creating a new version of the Delta table.

This demonstrates that Delta Lake supports in-place updates while preserving previous versions of the data.

The updated values will be reflected in the current version of the table, while older versions remain accessible via time travel.

### Verifying Delta Table Version History

After performing the update operation, we inspect the Delta table history again to confirm that a new version has been created.

Each modification to a Delta table results in a new version being recorded in the transaction log.

This allows us to track changes over time and enables features such as time travel and auditing.

In [0]:
spark.sql(
    """
    DESCRIBE HISTORY week4_catalog.default.tips_delta
    """
).show(truncate=False)

+-------+-------------------+--------------+-----------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName               |operation                        |operationParameters                                                                     

### Delta Versioning and Update Behavior

The DESCRIBE HISTORY output shows that the table now has two versions:

- Version 0: Initial table creation
- Version 1: Update operation applied to all rows

Key observations:
- The update operation modified 244 rows
- Delta rewrote the underlying data file (1 file removed, 1 file added)
- Previous data was not overwritten, but preserved as version 0

This demonstrates that Delta Lake:
- tracks all changes as new versions
- maintains a full history of the dataset
- enables safe and auditable data updates

This versioning capability is a key advantage over traditional file formats like Parquet.

### Comparing Different Versions Using Time Travel

In this step, we compare two versions of the Delta table:

- Version 0 → original data
- Version 1 → updated data (tip increased by 10%)

This demonstrates how Delta Lake allows us to access historical and current versions of the same dataset.

In [0]:
# Version 0 (original)
df_v0 = spark.sql(
    """
    SELECT tip 
    FROM week4_catalog.default.tips_delta
    VERSION AS OF 0
    """
)

# Version1 (current)
df_v1 = spark.sql(
    """
    SELECT tip 
    FROM week4_catalog.default.tips_delta
    """
)

df_v0.show(5)
df_v1.show(5)

+----+
| tip|
+----+
|1.01|
|1.66|
| 3.5|
|3.31|
|3.61|
+----+
only showing top 5 rows
+------------------+
|               tip|
+------------------+
|1.1110000000000002|
|             1.826|
|3.8500000000000005|
|3.6410000000000005|
|             3.971|
+------------------+
only showing top 5 rows


### Time Travel Comparison

By comparing version 0 and version 1 of the Delta table, we observe that:

- The `tip` values in version 1 are 10% higher than in version 0
- This reflects the update operation applied to the dataset

This demonstrates that:
- Delta Lake preserves previous versions of the data
- Updates do not overwrite existing data but create new versions
- Historical data remains accessible and queryable

This capability is critical for:
- debugging data pipelines
- auditing changes
- recovering from incorrect updates

### Writing Partitioned Delta Table

In this step, we write the dataset as a Delta table partitioned by the `day` column.

Partitioning organizes data into separate folders based on column values, which improves query performance by reducing the amount of data scanned.

The `day` column is a suitable partition key because it has low cardinality and is frequently used in analytical queries.

In [0]:
df_with_tip_pct.write.mode("overwrite").format("delta").partitionBy("day").saveAsTable("week4_catalog.default.tips_delta_partitioned")

### Inspecting Partition Structure

We verify that the Delta table is partitioned by the `day` column.

The SHOW PARTITIONS command lists all partitions, confirming that the data is physically organized into separate folders based on the partition key.

This validates that partitioning has been applied correctly and will improve query performance by enabling partition pruning.

In [0]:
spark.sql(
    """
    SHOW PARTITIONS week4_catalog.default.tips_delta_partitioned
    """
).show(truncate=False)

+----+
|day |
+----+
|Sat |
|Fri |
|Sun |
|Thur|
+----+



### Partition Pruning

Partition pruning is an optimization technique used by Spark to reduce the amount of data scanned during a query.

When a table is partitioned, data is physically stored in separate directories based on the partition column values (e.g., day=Sun, day=Sat).

If a query includes a filter on the partition column, Spark can:
- identify which partitions are relevant
- skip reading all other partitions

For example, when filtering:
WHERE day = 'Sun'

Spark will only read data from the `day=Sun` partition instead of scanning the entire dataset.

This significantly improves performance, especially for large datasets, by minimizing I/O and computation.

In [0]:
spark.sql(
    """
    SELECT *
    FROM week4_catalog.default.tips_delta_partitioned
    WHERE day = 'Sun'
    """
).show()

+----------+----+------+------+---+------+----+-------------------+
|total_bill| tip|   sex|smoker|day|  time|size|            tip_pct|
+----------+----+------+------+---+------+----+-------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|0.05944673337257211|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|0.16054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|0.16658733936220846|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2| 0.1397804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|0.14680764538430255|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|0.18623962040332148|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|0.22805017103762829|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|0.11607142857142858|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|0.13031914893617022|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2| 0.2185385656292287|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2| 0.1665043816942551|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|0

### Partition Pruning in Practice

The query filters data by `day = 'Sun'`, which matches one partition.

Because of partition pruning:
- Spark reads only the `day=Sun` partition
- Other partitions (`Sat`, `Fri`, `Thur`) are skipped

This reduces the amount of data scanned and improves query performance.

Partition pruning becomes especially valuable when working with large-scale datasets.

### Creating JSON Dataset

In this step, we convert the existing dataset into JSON format and store it in the volume.

This simulates working with multiple data formats in a real data pipeline, where data may come from different sources in formats such as CSV, JSON, or Parquet.

Having the same dataset in multiple formats allows us to compare their performance and characteristics.

In [0]:
df_with_tip_pct.write.mode("overwrite").json(
    "/Volumes/week4_catalog/default/week4_volume/json_output"
)

### Reading JSON Data

In this step, we read the dataset stored in JSON format.

JSON is a semi-structured, row-based format commonly used in APIs and log data. Compared to columnar formats like Parquet, JSON is more flexible but less efficient for analytical workloads.

We load the dataset to compare its structure and performance with other formats.

In [0]:
json_df = spark.read.json(
    "/Volumes/week4_catalog/default/week4_volume/json_output"
)
json_df.show()
json_df.printSchema()

+---+------+----+------+------+----+-------------------+----------+
|day|   sex|size|smoker|  time| tip|            tip_pct|total_bill|
+---+------+----+------+------+----+-------------------+----------+
|Sun|Female|   2|    No|Dinner|1.01|0.05944673337257211|     16.99|
|Sun|  Male|   3|    No|Dinner|1.66|0.16054158607350097|     10.34|
|Sun|  Male|   3|    No|Dinner| 3.5|0.16658733936220846|     21.01|
|Sun|  Male|   2|    No|Dinner|3.31| 0.1397804054054054|     23.68|
|Sun|Female|   4|    No|Dinner|3.61|0.14680764538430255|     24.59|
|Sun|  Male|   4|    No|Dinner|4.71|0.18623962040332148|     25.29|
|Sun|  Male|   2|    No|Dinner| 2.0|0.22805017103762829|      8.77|
|Sun|  Male|   4|    No|Dinner|3.12|0.11607142857142858|     26.88|
|Sun|  Male|   2|    No|Dinner|1.96|0.13031914893617022|     15.04|
|Sun|  Male|   2|    No|Dinner|3.23| 0.2185385656292287|     14.78|
|Sun|  Male|   2|    No|Dinner|1.71| 0.1665043816942551|     10.27|
|Sun|Female|   4|    No|Dinner| 5.0|0.1418037436

### JSON vs Parquet Schema Comparison

The JSON dataset has the same columns as the Parquet dataset, confirming that the data was written correctly.

However, a small difference is observed in data types:
- The `size` column is inferred as `long` in JSON
- In Parquet, it was stored as `integer`

This highlights an important distinction:
- JSON is a schema-less format where data types are inferred at read time
- Parquet preserves the schema explicitly

This difference demonstrates why columnar formats like Parquet are preferred for analytical workloads.

### Performance Comparison: CSV vs JSON vs Parquet

To compare performance across formats, we execute a full scan using the `count()` operation.

This triggers computation in Spark and allows us to observe differences in execution time between:
- CSV (row-based, no compression)
- JSON (semi-structured, flexible but less efficient)
- Parquet (columnar, optimized for analytics)

This comparison highlights why Parquet is preferred for large-scale data processing.

### Note on Dataset Consistency

The datasets used for performance comparison are slightly different:

- The CSV dataset represents the original data
- The JSON and Parquet datasets include an additional derived column (`tip_pct`)

While this introduces a minor difference in schema, the overall structure and size of the dataset remain comparable.

For precise benchmarking in production systems, it is recommended to compare datasets with identical schemas. However, for this analysis, the comparison remains valid for understanding general performance characteristics of different file formats.

In [0]:
import time

# CSV
start = time.time()
spark.read.csv(
    "/Volumes/week4_catalog/default/week4_volume/sales_data.csv",
    header=True,
    inferSchema=True
).count()
csv_time = time.time() - start

#JSON
start = time.time()
spark.read.json(
    "/Volumes/week4_catalog/default/week4_volume/json_output"
).count()
json_time = time.time() - start

# Parquet
start = time.time()
spark.read.parquet(
    "/Volumes/week4_catalog/default/week4_volume/parquet_output"
).count()
parquet_time = time.time() - start

print("CSV time:", csv_time)
print("JSON time:", json_time)
print("Parquet time:", parquet_time)

CSV time: 1.7710366249084473
JSON time: 1.3946342468261719
Parquet time: 1.209068775177002


### Performance Comparison Results

The execution times for the count operation are:

- CSV: ~1.77 seconds
- JSON: ~1.39 seconds
- Parquet: ~1.20 seconds

Observations:
- Parquet is the fastest format due to its columnar storage and compression
- JSON performs better than CSV but is still slower due to parsing overhead
- CSV is the slowest because it is a row-based format with no schema or optimization

Conclusion:
- Columnar formats like Parquet are significantly more efficient for analytical workloads
- Choosing the right file format is critical for performance in data engineering pipelines

### Generating a Spark Job for UI Analysis

In this step, we run a groupBy aggregation query to generate a Spark job.

This type of operation is useful for analyzing execution in the Spark UI because it:
- involves transformations and actions
- may trigger shuffles
- creates multiple stages in the execution plan

We will use this query to inspect how Spark processes distributed data.

In [0]:
from pyspark.sql.functions import avg, count 

spark.sql(
    """
    SELECT 
      day, 
      count(*) as transactions, 
      avg(total_bill) as avg_bill 
    FROM week4_catalog.default.tips_delta
    GROUP BY day
    """
).show()

+----+------------+------------------+
| day|transactions|          avg_bill|
+----+------------+------------------+
| Sun|          76|21.410000000000004|
| Sat|          87|20.441379310344825|
|Thur|          62|17.682741935483865|
| Fri|          19|17.151578947368417|
+----+------------+------------------+



### Spark UI Analysis: DAG and Execution Plan

The query execution was analyzed using the Spark UI (Query Profile).

Observations:
- The DAG (Directed Acyclic Graph) represents the logical execution plan of the query
- Each node in the DAG corresponds to a transformation step such as scanning data, projecting columns, or aggregating results
- The presence of an "Exchange" operation indicates a shuffle

Key insight:
- The GROUP BY operation triggered a shuffle, where data is redistributed across partitions based on the grouping key (`day`)
- This results in multiple execution stages

Conclusion:
- Spark breaks down queries into stages connected by shuffles
- Understanding the DAG helps identify performance bottlenecks, especially expensive operations like shuffles

### Shuffle Operation Explanation

A shuffle occurs when Spark redistributes data across partitions.

In this query, the GROUP BY operation requires all rows with the same `day` value to be processed together. To achieve this, Spark moves data across partitions so that records with the same key end up in the same partition.

This process is known as a shuffle and is one of the most expensive operations in Spark because it involves:
- data movement across the network
- disk I/O
- repartitioning of data

Understanding when shuffles occur is critical for optimizing Spark jobs.

### Lazy Evaluation in Spark

Spark uses lazy evaluation, meaning that transformations are not executed immediately.

Instead, Spark builds a logical execution plan (DAG) of transformations, and only executes them when an action is called.

Key concepts:
- **Transformations** (e.g., select, filter, groupBy) are lazy and only define the computation
- **Actions** (e.g., show, count, write) trigger execution

In this project:
- Transformations were applied when creating and modifying DataFrames
- Execution only occurred when actions like `.show()` or `.count()` were called

Benefits of lazy evaluation:
- Enables optimization of the execution plan
- Reduces unnecessary computations
- Improves overall performance

### Legacy SQL Migration to Spark SQL

In this step, we rewrite SQL queries similar to those used in Week 1 into Spark SQL.

This demonstrates how traditional relational queries can be executed in a distributed environment using Spark.

Spark SQL allows us to:
- use familiar SQL syntax
- execute queries on large datasets
- integrate seamlessly with DataFrame operations

In [0]:
spark.sql("""
SELECT 
    day,
    time,
    COUNT(*) AS total_transactions,
    AVG(total_bill) AS avg_bill,
    AVG(tip) AS avg_tip
FROM week4_catalog.default.tips_delta
WHERE total_bill > 10
GROUP BY day, time
ORDER BY avg_bill DESC
""").show()

+----+------+------------------+------------------+------------------+
| day|  time|total_transactions|          avg_bill|           avg_tip|
+----+------+------------------+------------------+------------------+
| Sun|Dinner|                71| 22.28056338028169|3.6154366197183085|
| Sat|Dinner|                83| 21.09385542168674| 3.386277108433736|
| Fri|Dinner|                11|20.928181818181816|             3.428|
|Thur|Dinner|                 1|             18.78|3.3000000000000003|
|Thur| Lunch|                55| 18.67854545454545| 3.188600000000001|
| Fri| Lunch|                 6|13.556666666666667|             2.706|
+----+------+------------------+------------------+------------------+



### Migration Observations

The query structure remains the same as in traditional SQL:
- SELECT, WHERE, GROUP BY, ORDER BY are used identically

Key difference:
- In Spark SQL, the query is executed in a distributed manner across partitions
- Operations such as GROUP BY trigger shuffles to aggregate data

This demonstrates that Spark SQL enables seamless migration from traditional SQL while scaling to large datasets.